## 1. `SliceMover.ipynb`
 
### Purpose
Given a polynomial system and a set of points lying on
`system ∩ start_slice`, `SliceMover` deforms the linear slice from
`start_slice` to `end_slice` via a continuation homotopy, tracking each
witness point to its corresponding point on `system ∩ end_slice`.
 
### Dependencies
```python
import bertini as b2
from bertini import nag_algorithm
from bertini import multiprec
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sympy
```

In [21]:
import bertini as b2
from bertini import nag_algorithm
from bertini import multiprec
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import sympy

#import eigenpy #to find mpfr precision

### Class `SliceMover`
 
```python
SliceMover(new_sys, start_slice, end_slice, start_points,
           new_t_start=1, new_t_end=0, new_gamma=None)
```
 
| Parameter | Meaning |
|---|---|
| `new_sys` | The polynomial system being intersected with the moving slice. |
| `start_slice`, `end_slice` | Either a `Slice` object (must expose `.as_system()`) or an already-built `System`. Handled polymorphically by `slice_to_sys`. |
| `start_points` | Points on `new_sys ∩ start_slice`, to be tracked as the slice moves. |
| `new_t_start`, `new_t_end` | Homotopy path-parameter endpoints (default full range `1 → 0`). |
| `new_gamma` | Optional fixed gamma constant for the gamma-trick (used to avoid path crossings/singularities during tracking). If omitted, a random rational gamma is drawn via `b2.symbolics.Rational.rand()`. |
 
**Constructor behavior:** stores the system and points, converts both
slices to system form, initializes a path-tracking variable
`self.tracking_var = b2.Variable('t')`, and immediately builds the
homotopy by calling `set_up_homotopy()`.
 
#### Methods
 
- **`slice_to_sys(start_slice, end_slice)`**
  Tries `slice.as_system()` on each argument; if that fails (e.g. the
  argument is already a `System`, not a `Slice`), falls back to using it
  as-is. Sets `self.start_sys` / `self.end_sys`.
- **`set_sys(new_sys)`**
  Replaces `self.system` and rebuilds the homotopy.
- **`set_start_slice(start_slice)`** / **`set_end_slice(end_slice)`**
  Update one side of the slice pair, re-derive `start_sys`/`end_sys`, and
  rebuild the homotopy. `set_end_slice` is the one reused in
  `RegenCascade.ipynb` to chain multiple slice moves in sequence.
- **`set_up_homotopy()`**
  Builds `self.hom = nag_algorithm.moving_homotopy(self.system, self.start_sys, self.end_sys, gamma=self.gamma)` and attaches the tracking
  variable via `self.hom.add_path_variable(self.tracking_var)`. This is
  the homotopy `H(x, t)` that literally moves the slice from start to end.
- **`move_slice()`**
  Builds the true target system as
  `b2.system.concatenate(self.system, self.end_sys)`, constructs a
  `b2.HomotopySolver(self.hom, self.start_points, target)`, solves it, and
  stores/returns the result (`self.results`).
- **`__repr__`**
  Prints the system and both slice-systems for debugging.

In [ ]:
class SliceMover():
    def __init__(self, new_sys, start_slice, end_slice, start_points, new_t_start = 1, new_t_end = 0, new_gamma = None):
        self.system = new_sys.clone() #copies so changes outside of SliceMover affect it
        self.start_sys = None
        self.end_sys = None
        
        self.slice_to_sys(start_slice, end_slice)

        self.hom = None
        
        self.start_points = start_points
        self.results = None
        
        if new_gamma == None:
            self.gamma = b2.symbolics.Rational.rand()
        else:
            self.gamma = new_gamma
        
        self.t_start = new_t_start
        self.t_end = new_t_end
        self.tracking_var = b2.Variable('t')
        self.set_up_homotopy()

    def slice_to_sys(self, start_slice, end_slice):
        # print("slice to sys")
        try:
            self.start_sys = start_slice.as_system()
        except:
            self.start_sys = start_slice

        try:
            self.end_sys = end_slice.as_system()
        except:
            self.end_sys = end_slice

    def set_sys(self, new_sys):
        self.system = new_sys
        self.set_up_homotopy()

    def set_start_slice(self, start_slice):
        self.start_slice = start_slice
        self.slice_to_sys(start_slice, self.end_sys)
        self.set_up_homotopy()

    def set_end_slice(self, end_slice):
        self.end_slice = end_slice
        self.slice_to_sys(self.start_sys, end_slice)
        self.set_up_homotopy()
    
    def __repr__(self):
        s = "system:\n" + str(self.system)
        s += "\nstart slice:\n" + str(self.start_sys)
        s += "\nend slice:\n" + str(self.end_sys)
        return s

    """
    Creates a moving_homotopy, stored in self.hom
    """
    def set_up_homotopy(self):
        # print("setting up hom")
        self.hom = nag_algorithm.moving_homotopy(self.system, self.start_sys, self.end_sys, gamma=self.gamma)
        self.hom.add_path_variable(self.tracking_var)

    """
    Solves the slice move using user_homotopy. Results are returned and also stored in self.results
    """
    def move_slice(self):
        # print("setting up target")
        target = b2.system.concatenate(self.system, self.end_sys)
        # print("creating mover")
        mover = b2.HomotopySolver(self.hom, self.start_points, target)
        # print("solving hom", self.hom)
        self.results = mover.solve()
        # print("returning")
        return self.results

### Demo cell (Lemniscate of Gerono)
 
The notebook includes a worked 2-variable example:
 
- Curve: `f = x**4 - x**2 + y**2` (Lemniscate of Gerono).
- Two lines: `l = 2x - 7y - 1` (start slice) and `m = x - 3y` (end slice).
- Start witness points are computed independently and directly, by
  solving `f ∩ l` with `nag_algorithm.ZeroDimSolver(..., mptype='adaptive')`
  and taking `.finite_solutions()` — this does **not** use `SliceMover`
  itself, it's just how the initial points for the demo are obtained.
- `SliceMover(sys, start_slice, end_slice, start_points_results)` is built
  and `.move_slice()` is called, tracking each point on `f ∩ l` to its
  counterpart on `f ∩ m`.

In [23]:
# driver

# uncomment below for demo
"""
import bertini as b2
import numpy as np
from bertini import nag_algorithm
from bertini import multiprec

x,y = b2.Variable('x'), b2.Variable('y')
vars = b2.VariableGroup([x, y])
#Lemniscate of Gerono system
f = x**4 - x**2 + y**2

#intersection lines from l to m
l = 2 * x - 7 * y - 1
m = x - 3 * y

#system setup
sys = b2.System()
sys.add_variable_group(vars)

start_slice = b2.system.clone(sys)
end_slice = b2.system.clone(sys)

sys.add(f)
start_slice.add(l)
end_slice.add(m)

start_points_sys = b2.system.clone(sys)
start_points_sys.add(l)
start_points = nag_algorithm.ZeroDimSolver(start_points_sys, mptype='adaptive')
start_points.solve()
start_points_results = start_points.finite_solutions()

sm = SliceMover(sys, start_slice, end_slice, start_points_results)
results = sm.move_slice()


print(results)
print(results.solutions)
s = results.solutions[0]
print(s)
"""

### Plotting utilities
 
- **`_eval_slice_y(slice_func, x_range)`** — parses the printed string form
  of a linear slice equation with `sympy`, extracts the `x`/`y`
  coefficients and constant term, and returns `y = -(a·x + c)/b` over
  `x_range` (returns `NaN` if the line is ~vertical, i.e. `b ≈ 0`).
- **`plot_slice_move(sm, results, curve_plotter, slice1_expr=None, slice2_expr=None)`**
  — draws the background curve (via `curve_plotter`), both slice lines
  (dashed blue/red), the start points (blue dots) and result points (red
  stars), with arrows from each start point to its tracked endpoint.
- **`lemniscate_plotter(ax)`**, **`circle_plotter(ax)`** — parametric plots
  of the Lemniscate of Gerono and unit circle, used as backgrounds.
- The actual call, `plot_slice_move(sm, results, lemniscate_plotter)`, is
  left commented out ("uncomment to see graph").

In [24]:
#graph driver

#uncomment to see graph (requires driver cell to run first)
"""
def _eval_slice_y(slice_func, x_range):
    s = str(slice_func).split('=')[-1].strip().replace(' ', '')
    x, y = sympy.symbols('x y')
    expr = sympy.sympify(s.replace('^', '**'))
    a = float(expr.coeff(x))
    b = float(expr.coeff(y))
    c = float(expr.subs([(x, 0), (y, 0)]))
    if abs(b) < 1e-12:
        return np.full_like(x_range, np.nan)
    return -(a * x_range + c) / b


def plot_slice_move(sm, results, curve_plotter, slice1_expr=None, slice2_expr=None):
    fig, ax = plt.subplots(figsize=(8, 8))
    curve_plotter(ax)

    def real_xy(sol):
        return float(complex(sol[0]).real), float(complex(sol[1]).real)

    start_points = sm.start_points
    start_xys = [real_xy(s) for s in start_points]
    result_xys = [real_xy(r) for r in results]

    all_xs = [x for x, y in start_xys + result_xys]
    margin = 0.3
    x_range = np.linspace(min(all_xs) - margin, max(all_xs) + margin, 200)

    label1 = slice1_expr or 'Start slice'
    y_start = _eval_slice_y(sm.start_sys, x_range)
    ax.plot(x_range, y_start, 'b--', linewidth=1.5, label=label1)

    label2 = slice2_expr or 'End slice'
    y_end = _eval_slice_y(sm.end_sys, x_range)
    ax.plot(x_range, y_end, 'r--', linewidth=1.5, label=label2)

    for sx, sy in start_xys:
        ax.plot(sx, sy, 'bo', markersize=8)

    for (sx, sy), (rx, ry) in zip(start_xys, result_xys):
        ax.plot(rx, ry, 'r*', markersize=10)
        ax.annotate('', xy=(rx, ry), xytext=(sx, sy),
                    arrowprops=dict(arrowstyle='->', color='gray',
                                   lw=1.2, connectionstyle='arc3,rad=0.3'))

    curve_line = next((l for l in ax.get_lines() if l.get_label() != '_nolegend_'), None)
    legend_handles = []
    if curve_line:
        legend_handles.append(plt.Line2D([0], [0], color=curve_line.get_color(),
                                         lw=2, label=curve_line.get_label()))
    legend_handles += [
        plt.Line2D([0], [0], color='b', lw=1.5, ls='--', label=label1),
        plt.Line2D([0], [0], color='r', lw=1.5, ls='--', label=label2),
        mpatches.Patch(color='blue', label='Start points'),
        mpatches.Patch(color='red',  label='End points'),
    ]
    ax.legend(handles=legend_handles)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title('Slice Move')
    ax.set_aspect('equal')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.axvline(0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def lemniscate_plotter(ax):
    t = np.linspace(0, 2 * np.pi, 1000)
    x = np.sin(t)
    y = np.sin(t) * np.cos(t)
    ax.plot(x, y, 'k-', linewidth=2, label='Lemniscate of Gerono')

def circle_plotter(ax):
    t = np.linspace(0, 2 * np.pi, 1000)
    ax.plot(np.cos(t), np.sin(t), 'k-', linewidth=2, label='Unit circle')


# --- usage ---
plot_slice_move(sm, results, lemniscate_plotter)
"""